In [3]:
import sqlite3
import glob

def merge_databases(output_db: str, db_pattern: str = "Top_Stats_*.db") -> None:
    master = sqlite3.connect(output_db)
    mcur = master.cursor()

    # Ensure master table exists (based on your current schema)
    mcur.execute('''CREATE TABLE IF NOT EXISTS player_stats (
        date_name_prof TEXT UNIQUE, date TEXT, year TEXT, month TEXT, day TEXT, num_fights REAL, duration REAL, account TEXT, guild_status TEXT, name TEXT, profession TEXT,
        damage REAL, down_contribution REAL, downs REAL, kills REAL, damage_taken REAL, damage_barrier REAL, downed REAL, deaths REAL, cleanses REAL,
        boon_strips REAL, resurrects REAL, healing REAL, barrier REAL, downed_healing REAL, stab_gen REAL, migh_gen REAL, fury_gen REAL,
        quic_gen REAL, alac_gen REAL, prot_gen REAL, rege_gen REAL, vigo_gen REAL, aeg_gen REAL, swif_gen REAL, resi_gen REAL, reso_gen REAL)''')

    # Get master column list
    mcur.execute("PRAGMA table_info(player_stats)")
    master_cols = [row[1] for row in mcur.fetchall()]
    master_colset = set(master_cols)

    for db_file in glob.glob(db_pattern):
        if db_file == output_db:
            continue

        print(f"Merging {db_file} → {output_db}")
        src = sqlite3.connect(db_file)
        scur = src.cursor()

        # Get source column list
        scur.execute("PRAGMA table_info(player_stats)")
        source_cols = [row[1] for row in scur.fetchall()]

        # Find intersection
        common_cols = [col for col in source_cols if col in master_colset]

        if not common_cols:
            print(f"⚠️ No matching columns between {db_file} and master DB.")
            src.close()
            continue

        col_list = ",".join(common_cols)
        placeholders = ",".join("?" * len(common_cols))

        # Select only the common columns
        query = f"SELECT {col_list} FROM player_stats"
        for row in scur.execute(query):
            try:
                mcur.execute(
                    f"INSERT OR IGNORE INTO player_stats ({col_list}) VALUES ({placeholders})",
                    row
                )
            except sqlite3.Error as e:
                print(f"⚠️ Error inserting row from {db_file}: {e}")

        src.close()

    master.commit()
    master.close()
    print("Merge complete ✅")




In [4]:
merge_databases("Top_Stats_Master.db", "Top_Stats_*.db")

Merging Top_Stats_1.db → Top_Stats_Master.db
Merging Top_Stats_2.db → Top_Stats_Master.db
Merge complete ✅
